# Digital signatures

In this notebook, we show how to sign messages and files using the ECDSA algorithm.

Examples are taken from: https://pycryptodome.readthedocs.io/en/latest/src/signature/dsa.html

First, let's import the library and the necessary modules.

In [1]:
from Crypto.PublicKey import ECC
from Crypto.Signature import DSS
from Crypto.Hash import SHA256

## ECC key pair generation

First we need to generate a pair of (public, secret) keys. In our example, we use the well-known P-256 elliptic curve (https://std.neuromancer.sk/nist/P-256).

In [2]:
# Generate a new ECC key (P-256 curve)
key = ECC.generate(curve='P-256')

Then we write the keys into separate files. 

Notice that, typically, the ECC private key file is protected with a password, so that it is resistant to brute force attacks.

We don't do that in this example.

In [3]:
pk_file_path = "results/public.pem"
sk_file_path = "results/private.pem"

# Export keys
private_key = key.export_key(format='PEM')
public_key = key.public_key().export_key(format='PEM')

# Save to files
with open(pk_file_path, "wt") as f:
    f.write(public_key)
    
with open(sk_file_path, "wt") as f:
    f.write(private_key)

## Signing a message

A **sender** wants to sign a message using their private key.

We can achieve this through the following steps.

1. Load the private key from file.
2. Compute the cryptographic hash of the message.
3. Sign the message hash.
4. Save the result (signature) to a file.

In [4]:
sk_file_path = "results/private.pem"
signature_file_path = "results/message_signature.bin"

# This is the message we want to sign.
message = "Hello world!"

# First, we read the private key from the corresponding file.
private_key = ECC.import_key(open(sk_file_path).read())

# Then we compute the SHA-256 hash of the message
h = SHA256.new(message.encode('utf-8'))

# We sign the message hash using the private key.
signer = DSS.new(private_key, 'fips-186-3')
signature = signer.sign(h)

# We save the result to a file.
with open(signature_file_path, "wb") as f:
    f.write(signature)

The **receiver** can use the sender's public key to verify the authenticity of the received message.

To do this, they need to:

1) Load the public key (we load it from the previously created file).
2) Hash the received message.
3) Verify the signature using the message hash and public key.

In [5]:
received_message = "Hello world!" # Suppose that this is the received message.
sender_pk_file_path = "results/public.pem" # This is where the sender's private key is stored.
signature_file_path = "results/message_signature.bin" # This is where the signature is stored.

# Load the public key.
sender_public_key = ECC.import_key(open(sender_pk_file_path).read())

# Load the signature.
signature = open(signature_file_path, "rb").read()

# Compute the hash of the message.
h = SHA256.new(received_message.encode('utf-8'))

# Verify the signature using the public key and the message hash.
verifier = DSS.new(sender_public_key, 'fips-186-3')
try:
    verifier.verify(h, signature)
    print("The message is authentic.")
except ValueError:
    print("The message is not authentic.")

The message is authentic.


## Signing a file

Can we do the same for a text file?

Let's see the process of signing the file. More precisely, we sign a digest of the file.

In [9]:
input_file = "data/othello_TXT_FolgerShakespeare.txt"
sk_file_path = "results/private.pem"
signature_file_path = "results/othello_signature.bin"

# First, we read the private key from the corresponding file.
private_key = ECC.import_key(open(sk_file_path).read())

# Then we read the input file and compute the SHA-256 hash.
# We read the file in chunks.
h = SHA256.new()
chunk_size = 8192
with open(input_file, "rb") as f:
    while chunk := f.read(chunk_size):
        h.update(chunk)
print("The hash of the file is:", h.hexdigest())

# We sign the message hash.
signer = DSS.new(private_key, 'fips-186-3')
signature = signer.sign(h)

# We save the result to a file.
with open(signature_file_path, "wb") as f:
    f.write(signature)

The hash of the file is: 6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722


Suppose we received the file and the signature.

To verify the signature, we need to:

1) Obtain the sender's public key.
2) Load the file and hash it.
3) Verify the signature using the public key and the message hash.

In [11]:
received_file = "data/othello_TXT_FolgerShakespeare.txt" # This is the path of the received file.
pk_file_path = "results/public.pem" # This is where the sender's private key is stored.
signature_file_path = "results/othello_signature.bin" # This is where the signature is stored.

# Load the sender's public key.
sender_public_key = ECC.import_key(open(pk_file_path).read())

# Load the signature.
signature = open(signature_file_path, "rb").read()

# Load the file and compute its hash.
h = SHA256.new()
chunk_size = 8192
with open(received_file, "rb") as f:
    while chunk := f.read(chunk_size):
        h.update(chunk)
print("The hash of the file is:", h.hexdigest())

# Verify the signature using the public key and the message hash.
verifier = DSS.new(sender_public_key, 'fips-186-3')
try:
    verifier.verify(h, signature)
    print("The message is authentic.")
except ValueError:
    print("The message is not authentic.")

The hash of the file is: 6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722
The message is authentic.
